<a href="https://colab.research.google.com/github/nihemelandu/clickstream-purchase-intent/blob/main/notebooks/01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Mount Google Drive and verify file access

from google.colab import drive
drive.mount('/content/drive')

import os

# Define file paths
BASE_PATH = '/content/drive/My Drive/'
FILES = {
    'oct': os.path.join(BASE_PATH, '2019-Oct.csv'),
    'nov': os.path.join(BASE_PATH, '2019-Nov.csv')
}

# Verify files exist and check sizes
for name, path in FILES.items():
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024 ** 3)
        print(f"{name}: {path}")
        print(f"  Size: {size_gb:.2f} GB")
        print(f"  Exists: True\n")
    else:
        print(f"{name}: FILE NOT FOUND at {path}\n")

In [ ]:
# Cell 2 — Install DuckDB and import libraries

!pip install duckdb -q

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Connect to DuckDB — in-memory analytical database
con = duckdb.connect()

# Verify DuckDB version
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")
print("All libraries imported successfully")

In [ ]:
# Cell 3 — Initial data inspection

# Inspect schema and first few rows of October file
print("=" * 60)
print("OCTOBER 2019 — SCHEMA AND SAMPLE ROWS")
print("=" * 60)

oct_sample = con.execute("""
    SELECT *
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Oct.csv',
        sample_size = 10000
    )
    LIMIT 5
""").df()

print(oct_sample.to_string())
print(f"\nColumn names: {list(oct_sample.columns)}")
print(f"Data types:\n{oct_sample.dtypes}")

# Inspect schema and first few rows of November file
print("\n" + "=" * 60)
print("NOVEMBER 2019 — SCHEMA AND SAMPLE ROWS")
print("=" * 60)

nov_sample = con.execute("""
    SELECT *
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Nov.csv',
        sample_size = 10000
    )
    LIMIT 5
""").df()

print(nov_sample.to_string())
print(f"\nColumn names: {list(nov_sample.columns)}")
print(f"Data types:\n{nov_sample.dtypes}")

# Confirm schemas match between the two files
oct_cols = list(oct_sample.columns)
nov_cols = list(nov_sample.columns)

print("\n" + "=" * 60)
print("SCHEMA CONSISTENCY CHECK")
print("=" * 60)
if oct_cols == nov_cols:
    print("✓ Both files have identical column names and order")
else:
    print("⚠ Column mismatch between files")
    print(f"  October columns:  {oct_cols}")
    print(f"  November columns: {nov_cols}")

In [ ]:
# Cell 4 — Row counts and combined volume

print("=" * 60)
print("ROW COUNTS PER FILE")
print("=" * 60)

oct_count = con.execute("""
    SELECT COUNT(*) as row_count
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Oct.csv',
        sample_size = -1
    )
""").df()

nov_count = con.execute("""
    SELECT COUNT(*) as row_count
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Nov.csv',
        sample_size = -1
    )
""").df()

oct_rows = oct_count['row_count'].values[0]
nov_rows = nov_count['row_count'].values[0]
total_rows = oct_rows + nov_rows

print(f"October 2019:  {oct_rows:>15,.0f} rows")
print(f"November 2019: {nov_rows:>15,.0f} rows")
print(f"Combined:      {total_rows:>15,.0f} rows")
print(f"\nNovember is {(nov_rows / oct_rows):.2f}x larger than October")

print("\n" + "=" * 60)
print("EVENT TYPE DISTRIBUTION — OCTOBER")
print("=" * 60)

oct_events = con.execute("""
    SELECT
        event_type,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Oct.csv',
        sample_size = -1
    )
    GROUP BY event_type
    ORDER BY count DESC
""").df()

print(oct_events.to_string(index=False))

print("\n" + "=" * 60)
print("EVENT TYPE DISTRIBUTION — NOVEMBER")
print("=" * 60)

nov_events = con.execute("""
    SELECT
        event_type,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Nov.csv',
        sample_size = -1
    )
    GROUP BY event_type
    ORDER BY count DESC
""").df()

print(nov_events.to_string(index=False))

print("\n" + "=" * 60)
print("EVENT TYPE DISTRIBUTION — COMBINED")
print("=" * 60)

combined_events = con.execute("""
    SELECT
        event_type,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM (
        SELECT event_type FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT event_type FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    )
    GROUP BY event_type
    ORDER BY count DESC
""").df()

print(combined_events.to_string(index=False))

# Visualise combined event distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4C72B0', '#DD8452', '#55A868']

# Bar chart
axes[0].bar(
    combined_events['event_type'],
    combined_events['count'],
    color=colors
)
axes[0].set_title('Event counts by type (combined)')
axes[0].set_xlabel('Event type')
axes[0].set_ylabel('Count')
for i, (count, pct) in enumerate(
    zip(combined_events['count'], combined_events['percentage'])
):
    axes[0].text(
        i, count + total_rows * 0.005,
        f'{pct}%',
        ha='center', fontsize=11
    )

# Pie chart
axes[1].pie(
    combined_events['count'],
    labels=combined_events['event_type'],
    autopct='%1.2f%%',
    colors=colors,
    startangle=90
)
axes[1].set_title('Event type proportions (combined)')

plt.tight_layout()
plt.savefig('event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved as event_distribution.png")

In [ ]:
# Cell 5 — Missing value analysis

print("=" * 60)
print("MISSING VALUE ANALYSIS — OCTOBER")
print("=" * 60)

oct_missing = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) as event_time_nulls,
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) as event_type_nulls,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) as product_id_nulls,
        SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) as category_id_nulls,
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) as category_code_nulls,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) as brand_nulls,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as price_nulls,
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) as user_id_nulls,
        SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) as user_session_nulls
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Oct.csv',
        sample_size = -1
    )
""").df()

# Reshape for readability
oct_missing_t = oct_missing.T.reset_index()
oct_missing_t.columns = ['feature', 'count']
oct_missing_t['percentage'] = (
    oct_missing_t['count'] / oct_rows * 100
).round(2)
oct_missing_t = oct_missing_t[oct_missing_t['feature'] != 'total_rows']

print(oct_missing_t.to_string(index=False))

print("\n" + "=" * 60)
print("MISSING VALUE ANALYSIS — NOVEMBER")
print("=" * 60)

nov_missing = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) as event_time_nulls,
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) as event_type_nulls,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) as product_id_nulls,
        SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) as category_id_nulls,
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) as category_code_nulls,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) as brand_nulls,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as price_nulls,
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) as user_id_nulls,
        SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) as user_session_nulls
    FROM read_csv_auto(
        '/content/drive/My Drive/2019-Nov.csv',
        sample_size = -1
    )
""").df()

nov_missing_t = nov_missing.T.reset_index()
nov_missing_t.columns = ['feature', 'count']
nov_missing_t['percentage'] = (
    nov_missing_t['count'] / nov_rows * 100
).round(2)
nov_missing_t = nov_missing_t[nov_missing_t['feature'] != 'total_rows']

print(nov_missing_t.to_string(index=False))

print("\n" + "=" * 60)
print("MISSING VALUE SUMMARY — COMBINED")
print("=" * 60)

combined_missing = oct_missing_t.copy()
combined_missing['count'] = oct_missing_t['count'] + nov_missing_t['count']
combined_missing['percentage'] = (
    combined_missing['count'] / total_rows * 100
).round(2)

print(combined_missing.to_string(index=False))

# Visualise missing values
fig, ax = plt.subplots(figsize=(12, 5))

features_with_missing = combined_missing[combined_missing['count'] > 0]

bars = ax.barh(
    features_with_missing['feature'],
    features_with_missing['percentage'],
    color=['#DD8452' if p > 10 else '#4C72B0'
           for p in features_with_missing['percentage']]
)

ax.set_xlabel('Missing (%)')
ax.set_title('Missing value percentage by feature (combined dataset)')

for bar, pct in zip(
    bars,
    features_with_missing['percentage']
):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f'{pct:.2f}%',
        va='center'
    )

plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nChart saved as missing_values.png")

In [ ]:
# Cell 6 — Session-level statistics

# Redefine constants
oct_rows = 42_448_764
nov_rows = 67_501_979
total_rows = oct_rows + nov_rows

print("=" * 60)
print("SESSION-LEVEL STATISTICS — COMBINED")
print("=" * 60)

session_stats = con.execute("""
    WITH combined AS (
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    ),
    session_level AS (
        SELECT
            user_session,
            COUNT(*) as event_count,
            COUNT(DISTINCT product_id) as product_count,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as purchase_count,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END)
                as cart_count,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)
                as view_count,
            MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as has_purchase,
            EPOCH(MAX(event_time)) - EPOCH(MIN(event_time))
                as session_duration_secs,
            AVG(price) as avg_price,
            MAX(price) as max_price,
            MIN(event_time) as session_start,
            MAX(event_time) as session_end
        FROM combined
        WHERE user_session IS NOT NULL
        GROUP BY user_session
    )
    SELECT
        COUNT(*) as total_sessions,
        SUM(has_purchase) as sessions_with_purchase,
        COUNT(*) - SUM(has_purchase) as sessions_without_purchase,
        ROUND(SUM(has_purchase) * 100.0 / COUNT(*), 2)
            as purchase_session_rate_pct,
        ROUND(AVG(event_count), 2) as mean_events_per_session,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY event_count)
            as median_events_per_session,
        MAX(event_count) as max_events_per_session,
        ROUND(AVG(session_duration_secs), 2) as mean_session_duration_secs,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY session_duration_secs)
            as median_session_duration_secs,
        ROUND(AVG(view_count), 2) as mean_views_per_session,
        ROUND(AVG(cart_count), 4) as mean_carts_per_session,
        ROUND(AVG(purchase_count), 4) as mean_purchases_per_session,
        ROUND(AVG(product_count), 2) as mean_products_per_session,
        ROUND(AVG(avg_price), 2) as mean_avg_price_per_session,
        MAX(max_price) as max_price_in_data
    FROM session_level
""").df()

# Reshape for readability
session_stats_t = session_stats.T.reset_index()
session_stats_t.columns = ['statistic', 'value']
print(session_stats_t.to_string(index=False))

print("\n" + "=" * 60)
print("SESSION OUTCOME BREAKDOWN")
print("=" * 60)

total_sessions = session_stats['total_sessions'].values[0]
purchase_sessions = session_stats['sessions_with_purchase'].values[0]
no_purchase_sessions = session_stats['sessions_without_purchase'].values[0]
purchase_rate = session_stats['purchase_session_rate_pct'].values[0]

print(f"Total sessions:              {total_sessions:>12,.0f}")
print(f"Sessions with purchase:      {purchase_sessions:>12,.0f}  "
      f"({purchase_rate:.2f}%)")
print(f"Sessions without purchase:   {no_purchase_sessions:>12,.0f}  "
      f"({100 - purchase_rate:.2f}%)")
print(f"Class imbalance ratio:       "
      f"{no_purchase_sessions / purchase_sessions:.1f} : 1")

# Visualise session duration and event count distributions
# Using a sample for visualisation only
print("\nSampling sessions for distribution plots...")

session_sample = con.execute("""
    WITH combined AS (
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    ),
    session_level AS (
        SELECT
            user_session,
            COUNT(*) as event_count,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END)
                as cart_count,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)
                as view_count,
            MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as has_purchase,
            EPOCH(MAX(event_time)) - EPOCH(MIN(event_time))
                as session_duration_secs
        FROM combined
        WHERE user_session IS NOT NULL
        GROUP BY user_session
    )
    SELECT *
    FROM session_level
    USING SAMPLE 200000
""").df()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Event count distribution (capped at 50 for readability)
axes[0, 0].hist(
    session_sample[session_sample['event_count'] <= 50]['event_count'],
    bins=50,
    color='#4C72B0',
    edgecolor='white',
    linewidth=0.3
)
axes[0, 0].set_title('Event count per session (capped at 50)')
axes[0, 0].set_xlabel('Number of events')
axes[0, 0].set_ylabel('Number of sessions')

# Session duration distribution (capped at 3600 secs = 1 hour)
axes[0, 1].hist(
    session_sample[
        session_sample['session_duration_secs'] <= 3600
    ]['session_duration_secs'],
    bins=60,
    color='#55A868',
    edgecolor='white',
    linewidth=0.3
)
axes[0, 1].set_title('Session duration in seconds (capped at 1 hour)')
axes[0, 1].set_xlabel('Duration (seconds)')
axes[0, 1].set_ylabel('Number of sessions')

# Event count by purchase outcome
purchasing = session_sample[
    session_sample['has_purchase'] == 1
]['event_count'].clip(upper=50)
non_purchasing = session_sample[
    session_sample['has_purchase'] == 0
]['event_count'].clip(upper=50)

axes[1, 0].hist(
    non_purchasing,
    bins=50,
    alpha=0.6,
    color='#4C72B0',
    label='No purchase',
    edgecolor='white',
    linewidth=0.3,
    density=True
)
axes[1, 0].hist(
    purchasing,
    bins=50,
    alpha=0.6,
    color='#DD8452',
    label='Purchase',
    edgecolor='white',
    linewidth=0.3,
    density=True
)
axes[1, 0].set_title('Event count by purchase outcome (normalised)')
axes[1, 0].set_xlabel('Number of events')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# Cart count by purchase outcome
purchasing_cart = session_sample[
    session_sample['has_purchase'] == 1
]['cart_count'].clip(upper=10)
non_purchasing_cart = session_sample[
    session_sample['has_purchase'] == 0
]['cart_count'].clip(upper=10)

axes[1, 1].hist(
    non_purchasing_cart,
    bins=10,
    alpha=0.6,
    color='#4C72B0',
    label='No purchase',
    edgecolor='white',
    linewidth=0.3,
    density=True
)
axes[1, 1].hist(
    purchasing_cart,
    bins=10,
    alpha=0.6,
    color='#DD8452',
    label='Purchase',
    edgecolor='white',
    linewidth=0.3,
    density=True
)
axes[1, 1].set_title('Cart additions by purchase outcome (normalised)')
axes[1, 1].set_xlabel('Number of cart additions')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

plt.suptitle('Session-level distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('session_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as session_distributions.png")

In [ ]:
# Cell 7 — User-level statistics

# Redefine constants
oct_rows = 42_448_764
nov_rows = 67_501_979
total_rows = oct_rows + nov_rows

print("=" * 60)
print("USER-LEVEL STATISTICS — COMBINED")
print("=" * 60)

user_stats = con.execute("""
    WITH combined AS (
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    ),
    user_level AS (
        SELECT
            user_id,
            COUNT(DISTINCT user_session) as session_count,
            COUNT(*) as event_count,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as purchase_count,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END)
                as cart_count,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END)
                as view_count,
            MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as has_purchase,
            COUNT(DISTINCT product_id) as distinct_products,
            COUNT(DISTINCT category_id) as distinct_categories,
            COUNT(DISTINCT brand) as distinct_brands,
            SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END)
                as total_spent,
            AVG(CASE WHEN event_type = 'purchase' THEN price END)
                as avg_price_per_purchase
        FROM combined
        WHERE user_id IS NOT NULL
        GROUP BY user_id
    )
    SELECT
        COUNT(*) as total_users,
        SUM(has_purchase) as users_with_purchase,
        ROUND(SUM(has_purchase) * 100.0 / COUNT(*), 2)
            as pct_users_with_purchase,
        ROUND(AVG(session_count), 2) as mean_sessions_per_user,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY session_count)
            as median_sessions_per_user,
        MAX(session_count) as max_sessions_per_user,
        ROUND(AVG(event_count), 2) as mean_events_per_user,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY event_count)
            as median_events_per_user,
        ROUND(AVG(purchase_count), 4) as mean_purchases_per_user,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY purchase_count)
            as median_purchases_per_user,
        MAX(purchase_count) as max_purchases_per_user,
        ROUND(AVG(distinct_products), 2) as mean_distinct_products,
        ROUND(AVG(distinct_categories), 2) as mean_distinct_categories,
        ROUND(AVG(distinct_brands), 2) as mean_distinct_brands,
        ROUND(AVG(total_spent), 2) as mean_total_spent,
        MAX(total_spent) as max_total_spent,
        ROUND(AVG(avg_price_per_purchase), 2) as mean_avg_price_per_purchase
    FROM user_level
""").df()

user_stats_t = user_stats.T.reset_index()
user_stats_t.columns = ['statistic', 'value']
print(user_stats_t.to_string(index=False))

print("\n" + "=" * 60)
print("USER PURCHASE BEHAVIOUR BREAKDOWN")
print("=" * 60)

total_users = user_stats['total_users'].values[0]
users_with_purchase = user_stats['users_with_purchase'].values[0]
users_without_purchase = total_users - users_with_purchase
pct_with_purchase = user_stats['pct_users_with_purchase'].values[0]

print(f"Total unique users:           {total_users:>12,.0f}")
print(f"Users with at least 1 purchase: {users_with_purchase:>10,.0f}  "
      f"({pct_with_purchase:.2f}%)")
print(f"Users with no purchase:       {users_without_purchase:>12,.0f}  "
      f"({100 - pct_with_purchase:.2f}%)")

# Purchase concentration analysis
print("\n" + "=" * 60)
print("PURCHASE CONCENTRATION ANALYSIS")
print("=" * 60)

purchase_concentration = con.execute("""
    WITH combined AS (
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT * FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    ),
    user_purchases AS (
        SELECT
            user_id,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as purchase_count,
            SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END)
                as total_spent
        FROM combined
        WHERE user_id IS NOT NULL
        GROUP BY user_id
    )
    SELECT
        SUM(CASE WHEN purchase_count = 0 THEN 1 ELSE 0 END)
            as users_0_purchases,
        SUM(CASE WHEN purchase_count = 1 THEN 1 ELSE 0 END)
            as users_1_purchase,
        SUM(CASE WHEN purchase_count = 2 THEN 1 ELSE 0 END)
            as users_2_purchases,
        SUM(CASE WHEN purchase_count BETWEEN 3 AND 5 THEN 1 ELSE 0 END)
            as users_3_to_5_purchases,
        SUM(CASE WHEN purchase_count > 5 THEN 1 ELSE 0 END)
            as users_over_5_purchases,
        SUM(CASE WHEN purchase_count > 0 THEN total_spent ELSE 0 END)
            as total_revenue,
        SUM(CASE WHEN purchase_count > 5 THEN total_spent ELSE 0 END)
            as revenue_from_heavy_buyers
    FROM user_purchases
""").df()

conc = purchase_concentration
total_rev = conc['total_revenue'].values[0]
heavy_rev = conc['revenue_from_heavy_buyers'].values[0]

print(f"Users with 0 purchases:       "
      f"{conc['users_0_purchases'].values[0]:>10,.0f}")
print(f"Users with 1 purchase:        "
      f"{conc['users_1_purchase'].values[0]:>10,.0f}")
print(f"Users with 2 purchases:       "
      f"{conc['users_2_purchases'].values[0]:>10,.0f}")
print(f"Users with 3-5 purchases:     "
      f"{conc['users_3_to_5_purchases'].values[0]:>10,.0f}")
print(f"Users with >5 purchases:      "
      f"{conc['users_over_5_purchases'].values[0]:>10,.0f}")
print(f"\nTotal revenue:                "
      f"{total_rev:>10,.2f}")
print(f"Revenue from heavy buyers (>5): "
      f"{heavy_rev:>10,.2f}  "
      f"({heavy_rev / total_rev * 100:.1f}% of total)")

# Visualise user purchase distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Purchase count distribution per user
user_sample = con.execute("""
    WITH combined AS (
        SELECT user_id, event_type FROM read_csv_auto(
            '/content/drive/My Drive/2019-Oct.csv',
            sample_size = -1
        )
        UNION ALL
        SELECT user_id, event_type FROM read_csv_auto(
            '/content/drive/My Drive/2019-Nov.csv',
            sample_size = -1
        )
    ),
    user_purchases AS (
        SELECT
            user_id,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                as purchase_count,
            COUNT(DISTINCT
                CASE WHEN event_type = 'purchase'
                THEN user_id END) as has_purchase
        FROM combined
        WHERE user_id IS NOT NULL
        GROUP BY user_id
    )
    SELECT purchase_count
    FROM user_purchases
    WHERE purchase_count <= 10
    USING SAMPLE 100000
""").df()

axes[0].hist(
    user_sample['purchase_count'],
    bins=11,
    color='#4C72B0',
    edgecolor='white',
    linewidth=0.5
)
axes[0].set_title('Purchase count per user (capped at 10)')
axes[0].set_xlabel('Number of purchases')
axes[0].set_ylabel('Number of users')
axes[0].set_xticks(range(11))

# Purchase concentration bar chart
categories = [
    '0 purchases', '1 purchase', '2 purchases',
    '3-5 purchases', '>5 purchases'
]
values = [
    conc['users_0_purchases'].values[0],
    conc['users_1_purchase'].values[0],
    conc['users_2_purchases'].values[0],
    conc['users_3_to_5_purchases'].values[0],
    conc['users_over_5_purchases'].values[0]
]

colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8172B2']
bars = axes[1].bar(categories, values, color=colors, edgecolor='white')
axes[1].set_title('User distribution by purchase frequency')
axes[1].set_xlabel('Purchase frequency group')
axes[1].set_ylabel('Number of users')
axes[1].tick_params(axis='x', rotation=20)

for bar, val in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total_users * 0.002,
        f'{val:,.0f}',
        ha='center',
        fontsize=9
    )

plt.suptitle('User-level purchase behaviour', fontsize=14)
plt.tight_layout()
plt.savefig('user_purchase_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as user_purchase_distribution.png")